In [ ]:
# Import dependencies

import ast
import io
import json
import os
import re
from html import escape
import pandas as pd
from PIL import Image
import random

import ipywidgets as widgets
from IPython.display import display, clear_output
import sys
sys.path.append('/pfss/mlde/workspaces/mlde_wsp_PI_Kersting/am84fuxo/LlavaGuard')
from llavaguard_config import local_data_dir
from llavaguard.taxonomy.PEGI.PEGI_Graph import get_content_categories, get_content_super_categories, get_max_pegi_score, get_super_cats_from_subcats


In [ ]:
# Parse labels and normalize ratings

def parse_consistency_labels(value):
    """Accept lists, JSON/Python list strings, comma-separated labels, and NA."""
    if isinstance(value, (list, tuple)):
        labels = list(value)
    elif value is None or pd.isna(value):
        return []
    elif isinstance(value, str):
        value = value.strip()
        if not value:
            return []
        if value.startswith('['):
            labels = ast.literal_eval(value)
            if not isinstance(labels, list):
                raise ValueError('Expected a list of label strings')
        else:
            labels = value.split(',')
    else:
        raise ValueError('Expected labels as a string or list')
    if any(not isinstance(label, str) for label in labels):
        raise ValueError('Each label must be a string')
    return [label.strip() for label in labels if label.strip()]


def parse_categories(value):
    """Return category or subcategory labels without the explicit NA marker."""
    return [label for label in parse_consistency_labels(value)
            if label not in {"NA: None applying", "NA: None applicable"}]


def normalize_pegi(value):
    """Normalize valid PEGI ratings without truncating fractional values."""
    match = re.fullmatch(r"(?:PEGI\s*)?(3|7|12|16|18)(?:\.0+)?\+?", str(value).strip(), re.I)
    return int(match.group(1)) if match else None


def is_missing(value):
    """Distinguish missing annotations from explicit empty lists or NA."""
    if isinstance(value, (list, tuple)):
        return False
    return value is None or pd.isna(value) or (isinstance(value, str) and not value.strip())


def serialize_labels(value):
    """Store labels as JSON, preserving missing annotations as missing."""
    if is_missing(value):
        return None
    labels = parse_categories(value)
    return json.dumps(labels, ensure_ascii=False) if labels else "NA: None applying"


def image_path_of(sample):
    """Read the supported image-path fields."""
    return sample.get("image_path") or sample.get("image path") or sample.get("image")


def checked_subcategories(value):
    """Reject missing or unknown labels before deriving categories or ratings."""
    if is_missing(value):
        raise ValueError("Missing subcategories")
    raw = parse_consistency_labels(value)
    labels = parse_categories(value)
    unknown = set(labels) - set(get_content_categories())
    if unknown:
        raise ValueError(f"Unknown subcategories: {sorted(unknown)}")
    if labels and any(label in {"NA: None applying", "NA: None applicable"} for label in raw):
        raise ValueError("NA mixed with subcategories")
    return labels


def derived_categories(labels):
    """Return parent categories as a list, including for safe images."""
    return parse_categories(get_super_cats_from_subcats(labels))


In [ ]:
# Load remaining samples and choose a sampling mode

JSON_PATH = f"{local_data_dir}/datasets/PEGI-Guard-DS/v4/all_labels.json"
OUT_CSV  = f"{local_data_dir}/evaluation/human_validation/human_validation.csv"

with open(JSON_PATH, "r") as f:
    samples = json.load(f)

random.seed(42)
random.shuffle(samples)

# Exclude already validated samples.
already_validated = set()
if os.path.exists(OUT_CSV):
    try:
        validated_df = pd.read_csv(OUT_CSV)
        already_validated = set(validated_df["id"].astype(str))
        print(f"Found {len(already_validated)} already validated samples in {OUT_CSV}")
    except Exception as e:
        print(f"Warning: Could not read output CSV: {e}")

# Filter out already validated samples.
samples = [s for s in samples if str(s.get("id")) not in already_validated]
print(f"Remaining samples to validate: {len(samples)}")

SUBCATEGORIES = get_content_categories()


remaining_samples = list(samples)
SAMPLING_MODE = "categories"  # Choose "categories", "rating", or "all".
if SAMPLING_MODE not in {"categories", "rating", "all"}:
    raise ValueError("Unknown sampling mode")


In [ ]:
# Build a rating-guided validation queue

if SAMPLING_MODE == "rating":
    # Use automated ratings to target gaps in human rating coverage.

    from collections import Counter, defaultdict
    from llavaguard.taxonomy.PEGI.PEGI_Graph import get_max_pegi_score

    TARGET_PER_PEGI = 50
    PEGI_RATINGS = [3, 7, 12, 16, 18]

    # Count already validated human PEGI ratings.
    validated_counts = Counter()

    if os.path.exists(OUT_CSV):
        validated_df = pd.read_csv(OUT_CSV)

        if "PEGI rating" in validated_df.columns:
            for r in validated_df["PEGI rating"]:
                r = normalize_pegi(r)
                if r in PEGI_RATINGS:
                    validated_counts[r] += 1

    print("Already validated per (human) PEGI:", dict(validated_counts))

    # Bucket remaining samples by their automated PEGI rating.
    samples_by_pegi = defaultdict(list)

    for s in remaining_samples:
        pegi = normalize_pegi(s.get("PEGI rating"))
        if pegi in PEGI_RATINGS:
            samples_by_pegi[pegi].append(s)

    # Select samples to fill underrepresented PEGI levels first.
    balanced_samples = []

    for pegi in PEGI_RATINGS:
        needed = max(0, TARGET_PER_PEGI - validated_counts[pegi])
        candidates = samples_by_pegi[pegi]

        selected = candidates[:needed]
        balanced_samples.extend(selected)

        print(
            f"PEGI {pegi}: already={validated_counts[pegi]}, "
            f"needed={needed}, available={len(candidates)}, selected={len(selected)}"
        )

    random.seed(42)
    random.shuffle(balanced_samples)

    samples = balanced_samples

    print(f"Rating-guided validation queue size: {len(samples)}")


In [ ]:
# Build a category-guided validation queue

if SAMPLING_MODE == "categories":
    import ast


    # Use automated categories to target gaps in human category coverage.
    from collections import Counter, defaultdict

    TARGET_PER_CAT = 20
    CATEGORIES = get_content_super_categories()


    # Count categories already represented in human validation.
    validated_counts = Counter()

    if os.path.exists(OUT_CSV):
        validated_df = pd.read_csv(OUT_CSV)

        # Reconstruct human_categories if the old CSV is missing the column.
        if "human_categories" not in validated_df.columns:
            print("human_categories missing - reconstructing from human_subcategories...")

            validated_df["human_categories"] = validated_df["human_subcategories"].apply(
                lambda x: get_super_cats_from_subcats(parse_categories(x))
            )

            print("Reconstructed human_categories for sampling counts.")

        # Now count human categories.
        for value in validated_df["human_categories"]:
            cats = parse_categories(value)

            for cat in cats:
                if cat in CATEGORIES:
                    validated_counts[cat] += 1


    # Bucket remaining samples by automated category.
    samples_by_cat = defaultdict(list)

    for s in remaining_samples:
        for cat in parse_categories(s.get("categories")):
            if cat in CATEGORIES:
                samples_by_cat[cat].append(s)


    # Current counts include previously validated samples.
    current_counts = validated_counts.copy()

    balanced_samples = []
    selected_ids = set()

    for cat in CATEGORIES:
        needed = max(0, TARGET_PER_CAT - current_counts[cat])
        candidates = samples_by_cat[cat]

        selected_for_cat = []

        for s in candidates:
            if needed <= 0:
                break

            if s["id"] in selected_ids:
                continue

            selected_for_cat.append(s)
            balanced_samples.append(s)
            selected_ids.add(s["id"])

            # This image may contribute to multiple categories.
            sample_cats = parse_categories(s.get("categories"))

            for sample_cat in sample_cats:
                if sample_cat in CATEGORIES:
                    current_counts[sample_cat] += 1

            needed = max(0, TARGET_PER_CAT - current_counts[cat])

        print(
            f"{cat}: previously_validated={validated_counts[cat]}, "
            f"final_count={current_counts[cat]}, "
            f"available={len(candidates)}, "
            f"newly_selected_for_cat={len(selected_for_cat)}"
        )


    # Randomize validation order.
    random.seed(42)
    random.shuffle(balanced_samples)

    samples = balanced_samples

    print(f"\nBalanced category validation queue size: {len(samples)}")

    print("\nEstimated category coverage using automated labels:")
    for cat in CATEGORIES:
        print(f"{cat}: {current_counts[cat]}")


In [ ]:
# Review and save human annotations

class HumanValidationUI:
    def __init__(self, samples, subcategories, out_csv, image_size=(400, 400)):
        self.samples = samples
        self.subcategories = subcategories
        self.out_csv = out_csv
        self.image_size = image_size

        self.current_idx = 0

        # Store one row per reviewed sample.
        cols = ["id", "image_path", "human_categories", "human_subcategories", "PEGI rating", "auto_categories", "auto_subcategories", "auto PEGI rating", "notes"]
        self.human_feedback = pd.DataFrame(columns=cols)

        # Load existing feedback if available (for resuming validation).
        if os.path.exists(out_csv):
            try:
                self.human_feedback = pd.read_csv(out_csv)
                print(f"Loaded {len(self.human_feedback)} existing validation records from {out_csv}")
            except Exception as e:
                raise ValueError(f"Cannot safely resume from {out_csv}") from e

        # Widgets.
        self.image_widget = widgets.Image(format="png", width=image_size[0], height=image_size[1])
        self.header = widgets.HTML()
        self.output = widgets.Output()

        # Optional notes / rationale.
        self.notes_widget = widgets.Textarea(
            description="Notes",
            placeholder="Optional: rationale, uncertainty, issues with image, etc.",
            layout=widgets.Layout(width="98%", height="80px")
        )

        # Checkboxes for subcategories.
        self.checkbox_widgets = {
            sc: widgets.Checkbox(
                description=sc,
                value=False,
                indent=False,
                layout=widgets.Layout(margin="0px 12px 0px 0px", width="340px")
            )
            for sc in self.subcategories
        }

        # Buttons.
        self.prev_button = widgets.Button(description="← Prev", button_style="")
        self.next_button = widgets.Button(description="Next →", button_style="primary")
        self.save_button = widgets.Button(description="Save now", button_style="warning")
        self.finish_button = widgets.Button(description="Finish + Export", button_style="success")

        self.prev_button.on_click(self.prev_image)
        self.next_button.on_click(self.next_image)
        self.save_button.on_click(self.save_partial)
        self.finish_button.on_click(self.finish)

        # Layout: make checkbox grid (3 per row is usually readable with long labels).
        checkbox_rows = []
        row_size = 3
        sc_list = list(self.checkbox_widgets.values())
        for i in range(0, len(sc_list), row_size):
            checkbox_rows.append(widgets.HBox(sc_list[i:i+row_size]))

        self.controls = widgets.HBox([self.prev_button, self.next_button, self.save_button, self.finish_button])
        self.widget_box = widgets.VBox([
            self.header,
            self.output,
            widgets.VBox(checkbox_rows),
            self.notes_widget,
            self.controls
        ])

        display(self.image_widget)
        display(self.widget_box)
        self.load_current_sample()

    def _load_image_bytes(self, path):
        # Handles missing images gracefully.
        try:
            img = Image.open(path).convert("RGB").resize(self.image_size)
            buf = io.BytesIO()
            img.save(buf, format="PNG")
            return buf.getvalue(), None
        except Exception as e:
            return b"", str(e)

    def load_current_sample(self):
        clear_output(wait=True)
        display(self.image_widget)
        display(self.widget_box)

        if self.current_idx >= len(self.samples):
            self._export_and_done()
            return

        s = self.samples[self.current_idx]
        sid = s.get("id", f"idx_{self.current_idx}")
        ipath = image_path_of(s) or ""

        orig_cats = parse_categories(s.get("categories"))
        orig_subcats = parse_categories(s.get("subcategories"))

        # Header.
        self.header.value = (
            f"<b>Sample</b> {self.current_idx+1}/{len(self.samples)} "
            f"&nbsp;&nbsp; <b>ID:</b> {escape(str(sid))} "
            f"&nbsp;&nbsp; <b>Image:</b> {escape(str(ipath))}"
        )

        # Image.
        img_bytes, err = self._load_image_bytes(ipath)
        self.image_widget.value = img_bytes

        with self.output:
            self.output.clear_output()
            if err:
                print(f"[WARN] Could not load image: {err}")
            print("Original categories:", orig_cats)
            print("Original subcategories:", orig_subcats)

        # Restore human edits when revisiting a reviewed sample.
        saved = self.human_feedback[
            self.human_feedback["id"].astype(str) == str(sid)
        ]
        if saved.empty:
            self.notes_widget.value = ""
            orig_set = set(orig_subcats)
        else:
            previous = saved.iloc[-1]
            notes = previous.get("notes")
            self.notes_widget.value = "" if pd.isna(notes) else str(notes)
            orig_set = set(parse_categories(previous.get("human_subcategories")))
        for sc in self.subcategories:
            self.checkbox_widgets[sc].value = (sc in orig_set)

    def _current_human_subcategories(self):
        return [sc for sc, cb in self.checkbox_widgets.items() if cb.value]

    def save_current_feedback(self):
        if self.current_idx >= len(self.samples):
            return

        s = self.samples[self.current_idx]

        sid = s.get("id", f"idx_{self.current_idx}")
        ipath = image_path_of(s) or ""
        auto_cats = parse_categories(s.get("categories"))
        auto_subcats = parse_categories(s.get("subcategories"))
        auto_pegi_rating = s.get("PEGI rating")

        human_subcats = self._current_human_subcategories()
        human_cats = derived_categories(human_subcats)
        notes = self.notes_widget.value.strip()

        pegi_rating = get_max_pegi_score(human_subcats)

        row = {
            "id": sid,
            "image_path": ipath,
            "human_categories": serialize_labels(human_cats),
            "human_subcategories": serialize_labels(human_subcats),
            "PEGI rating": pegi_rating,
            "auto_categories": json.dumps(auto_cats, ensure_ascii=False),
            "auto_subcategories": json.dumps(auto_subcats, ensure_ascii=False),
            "auto PEGI rating": auto_pegi_rating,
            "notes": notes,
        }

        self.human_feedback = self.human_feedback.dropna(how="all")

        cols = [
            "id",
            "image_path",
            "human_categories",
            "human_subcategories",
            "PEGI rating",
            "auto_categories",
            "auto_subcategories",
            "auto PEGI rating",
            "notes",
        ]

        for col in cols:
            if col not in self.human_feedback.columns:
                self.human_feedback[col] = None

        self.human_feedback = self.human_feedback[cols]

        # Replace existing row for this id, otherwise append.
        mask = self.human_feedback["id"].astype(str) == str(sid)

        if mask.any():
            idx = self.human_feedback.index[mask][0]

            for col in cols:
                self.human_feedback.at[idx, col] = row[col]
        else:
            self.human_feedback = pd.concat(
                [self.human_feedback, pd.DataFrame([row])],
                ignore_index=True
            )

    def _write_feedback(self):
        """Save progress after each confirmed image."""
        os.makedirs(os.path.dirname(self.out_csv) or ".", exist_ok=True)
        self.human_feedback.to_csv(self.out_csv, index=False)

    def next_image(self, b):
        self.save_current_feedback()
        self._write_feedback()
        self.current_idx += 1
        self.load_current_sample()

    def prev_image(self, b):
        self.save_current_feedback()
        self.current_idx = max(0, self.current_idx - 1)
        self.load_current_sample()

    def save_partial(self, b):
        self.save_current_feedback()

        os.makedirs(os.path.dirname(self.out_csv) or ".", exist_ok=True)

        self.human_feedback = self.human_feedback.dropna(how="all")

        self.human_feedback.to_csv(self.out_csv, index=False)

        with self.output:
            print(f"[Saved] {len(self.human_feedback)} samples -> {self.out_csv}")

    def finish(self, b):
        self.save_current_feedback()
        self._export_and_done()

    def _export_and_done(self):
        os.makedirs(os.path.dirname(self.out_csv) or ".", exist_ok=True)
        self.human_feedback.to_csv(self.out_csv, index=False)

        clear_output()
        print("Validation complete!")
        print(f"Saved CSV:   {self.out_csv}")


In [ ]:
# Start the validation interface

ui = HumanValidationUI(
    samples=samples,
    subcategories=SUBCATEGORIES,
    out_csv=OUT_CSV,
    image_size=(400, 400)
)


## Check and clean annotations

Run cleanup cells as needed. They save changes to the configured CSV.


In [ ]:
# Check label consistency

import ast
import re
import pandas as pd
from llavaguard.taxonomy.PEGI.PEGI_Graph import policy_graph, get_max_pegi_score


def check_label_consistency(frame, sources=('human', 'auto')):
    """Return one row per issue; an empty report means all checked rows agree.

    Missing values are reported separately from explicit empty/NA labels.
    Empty/NA subcategories imply no categories and PEGI 3.
    This checks internal consistency, not whether labels describe the image.
    """
    columns = ['row_index', 'id', 'source', 'field', 'issue', 'actual', 'expected']
    issues = []
    na_labels = {'NA: None applying', 'NA: None applicable'}
    sub_to_cat = {sub: cat for cat, subs in policy_graph.items() for sub in subs}

    def report(index, sid, source, field, issue, actual, expected=None):
        issues.append(dict(zip(columns, [index, sid, source, field, issue, actual, expected])))

    for source in sources:
        if source not in ('human', 'auto'):
            raise ValueError(f'Unknown source: {source!r}')
        cat_col, sub_col = f'{source}_categories', f'{source}_subcategories'
        rating_col = 'PEGI rating' if source == 'human' else 'auto PEGI rating'
        required = [cat_col, sub_col, rating_col]
        for col in required:
            if col not in frame:
                report(None, None, source, col, 'missing_column', None)
        if any(col not in frame for col in required):
            continue

        for index, row in frame.iterrows():
            sid = row.get('id', index)
            parsed = {}
            for col, known in ((cat_col, set(policy_graph)), (sub_col, set(sub_to_cat))):
                value = row[col]
                if not isinstance(value, (list, tuple)) and (
                    value is None or pd.isna(value)
                    or (isinstance(value, str) and not value.strip())
                ):
                    report(index, sid, source, col, 'missing_labels', value)
                    continue
                try:
                    labels = parse_consistency_labels(value)
                except (ValueError, SyntaxError, TypeError) as error:
                    report(index, sid, source, col, 'invalid_labels', str(value), str(error))
                    continue
                if len(labels) != len(set(labels)):
                    report(index, sid, source, col, 'duplicate_labels', labels)
                real_labels = [label for label in labels if label not in na_labels]
                if real_labels and any(label in na_labels for label in labels):
                    report(index, sid, source, col, 'na_mixed_with_labels', labels)
                unknown = sorted(set(real_labels) - known)
                if unknown:
                    report(index, sid, source, col, 'unknown_labels', unknown)
                    continue
                parsed[col] = real_labels

            if sub_col not in parsed:
                continue
            subcategories = parsed[sub_col]
            expected_cats = sorted({sub_to_cat[sub] for sub in subcategories})
            if cat_col in parsed and set(parsed[cat_col]) != set(expected_cats):
                report(index, sid, source, cat_col, 'category_mismatch', parsed[cat_col], expected_cats)

            expected_rating = get_max_pegi_score(subcategories)
            value = row[rating_col]
            # Accept 12, 12.0, 'PEGI 12', and '12+' without truncating 12.5.
            match = re.fullmatch(r'(?:PEGI\s*)?(\d+(?:\.0+)?)\+?', str(value).strip(), re.I)
            actual_rating = int(float(match.group(1))) if match else None
            if actual_rating != expected_rating:
                report(index, sid, source, rating_col, 'rating_mismatch', value, expected_rating)

    return pd.DataFrame(issues, columns=columns)


In [ ]:
# Inspect annotation inconsistencies

OUT_CSV = f"{local_data_dir}/evaluation/human_validation/human_validation.csv"
validation_df = pd.read_csv(OUT_CSV)
consistency_issues = check_label_consistency(validation_df)
display(consistency_issues)


In [ ]:
# Filter and sort taxonomy tables

def filter_excluded_subcategories(df, excluded_numbers=(46, 47)):
    """
    Remove subcategories whose numeric prefix is in excluded_numbers.
    """
    def extract_number(label):
        match = re.match(r"(\d+)", str(label))
        return int(match.group(1)) if match else None

    mask = df.index.map(lambda x: extract_number(x) not in excluded_numbers)
    return df.loc[mask]

def filter_excluded_categories(df, excluded_number=9):
    """
    Remove categories whose S-number equals excluded_number,
    e.g. excluded_number=9 removes 'S9. ...'
    """
    def extract_s_number(label: str):
        if isinstance(label, str) and label.startswith("S"):
            match = re.match(r"S(\d+)", label)
            if match:
                return int(match.group(1))
        return None  # safer than 999 for filtering

    mask = df.index.map(lambda x: extract_s_number(x) != excluded_number)
    return df.loc[mask]

def sort_by_category_number(df):
    """
    Sort a DataFrame by the numeric prefix in the index, e.g.:
    'S1. Violence' = 1
    'S2. Hate' = 2
    """
    def extract_s_number(label: str):
        if label.startswith("S"):
            match = re.match(r"S(\d+)", label)
            if match:
                return int(match.group(1))
        return 999  # NA goes last
    df = df.copy()
    df["__num__"] = df.index.map(extract_s_number)
    df = df.sort_values("__num__", ascending=True)
    return df.drop(columns=["__num__"])

def sort_by_subcategory_number(df):
    """
    Sort a DataFrame by the numeric prefix in the index, e.g.:
    '1. Educational Content' = 1
    '45. High-Risk Financial Activities' = 45
    """
    def extract_number(label):
        if label.startswith("NA"):
            return 999 # NA goes last
        match = re.match(r"(\d+)", str(label))
        return int(match.group(1)) if match else float("inf")

    df = df.copy()
    df["__num__"] = df.index.map(extract_number)
    df = df.sort_values("__num__", ascending=True)
    return df.drop(columns=["__num__"])


In [ ]:
# Fill missing automated annotations

CSV_PATH = OUT_CSV
df = pd.read_csv(CSV_PATH)
with open(JSON_PATH, "r") as handle:
    auto_data = json.load(handle)
auto_by_image = {image_path_of(item): item for item in auto_data if image_path_of(item)}
updated = 0
unmatched = []
for col in ["auto_categories", "auto_subcategories", "auto PEGI rating"]:
    if col not in df:
        df[col] = None
for idx, row in df.iterrows():
    sample = auto_by_image.get(row.get("image_path"))
    if sample is None:
        unmatched.append(row.get("image_path"))
        continue
    changed = False
    for col, key in [("auto_categories", "categories"), ("auto_subcategories", "subcategories")]:
        if is_missing(row.get(col)) and not is_missing(sample.get(key)):
            df.at[idx, col] = serialize_labels(sample[key])
            changed = True
    if is_missing(row.get("auto PEGI rating")):
        try:
            labels = checked_subcategories(df.at[idx, "auto_subcategories"])
        except ValueError:
            continue
        df.at[idx, "auto PEGI rating"] = get_max_pegi_score(labels)
        changed = True
    updated += int(changed)
df.to_csv(CSV_PATH, index=False)
print(f"Updated rows: {updated}; unmatched images: {len(unmatched)}")


In [ ]:
# Fill missing human PEGI ratings

import pandas as pd
from llavaguard.taxonomy.PEGI.PEGI_Graph import get_content_categories, get_max_pegi_score

CSV_PATH = f"{local_data_dir}/evaluation/human_validation/human_validation.csv"
df = pd.read_csv(CSV_PATH)

missing_rating = df["PEGI rating"].isna() | df["PEGI rating"].astype(str).str.strip().eq("")
known_subcategories = set(get_content_categories())
na_labels = {"NA: None applying", "NA: None applicable"}
filled = 0
skipped = []

for idx in df.index[missing_rating]:
    value = df.at[idx, "human_subcategories"]
    # Missing annotations are different from an explicit [] or NA (safe).
    if not isinstance(value, (list, tuple)) and (
        value is None or pd.isna(value) or (isinstance(value, str) and not value.strip())
    ):
        skipped.append((idx, "Missing human subcategories"))
        continue
    try:
        labels = parse_consistency_labels(value)
        subcategories = [label for label in labels if label not in na_labels]
        unknown = set(subcategories) - known_subcategories
        if unknown:
            raise ValueError(f"Unknown subcategories: {sorted(unknown)}")
        if subcategories and any(label in na_labels for label in labels):
            raise ValueError("NA mixed with subcategories")
        df.at[idx, "PEGI rating"] = get_max_pegi_score(subcategories)
        filled += 1
    except (ValueError, SyntaxError, TypeError) as error:
        skipped.append((idx, str(error)))

if filled:
    df.to_csv(CSV_PATH, index=False)
print(f"Filled {filled} missing human PEGI ratings; existing ratings preserved.")
print(f"Skipped {len(skipped)} rows with missing or invalid subcategories.")
if skipped:
    display(pd.DataFrame(skipped, columns=["row_index", "reason"]))

# Refresh the consistency report after saving.
validation_df = df.copy()
consistency_issues = check_label_consistency(validation_df)
display(consistency_issues)


In [ ]:
# Rebuild human categories from subcategories

CSV_PATH = OUT_CSV
df = pd.read_csv(CSV_PATH)
if "human_categories" not in df:
    df["human_categories"] = None
skipped = []
for idx, value in df["human_subcategories"].items():
    try:
        labels = checked_subcategories(value)
    except ValueError as error:
        skipped.append((idx, str(error)))
        continue
    df.at[idx, "human_categories"] = serialize_labels(derived_categories(labels))
df.to_csv(CSV_PATH, index=False)
print(f"Rebuilt human categories; skipped {len(skipped)} missing or invalid annotations.")


In [ ]:
# Normalize label serialization

def normalize_labels(value):
    """Serialize label fields without turning missing annotations into safe labels."""
    return serialize_labels(value)


In [ ]:
# Refresh matched automated annotations

import json
import pandas as pd

CSV_PATH = f"{local_data_dir}/evaluation/human_validation/human_validation.csv"
JSON_PATH = f"{local_data_dir}/datasets/PEGI-Guard-DS/v4/all_labels.json"

df = pd.read_csv(CSV_PATH)

with open(JSON_PATH, "r") as f:
    auto_data = json.load(f)


auto_by_id = {str(item["id"]): item for item in auto_data if item.get("id") is not None}
matched = df["id"].astype(str).isin(auto_by_id)
for idx in df.index[matched]:
    sample = auto_by_id[str(df.at[idx, "id"])]
    for col, key in [("auto_categories", "categories"), ("auto_subcategories", "subcategories"),
                     ("auto PEGI rating", "PEGI rating")]:
        value = sample.get(key)
        if not is_missing(value):
            df.at[idx, col] = serialize_labels(value) if key != "PEGI rating" else value

for col in [
    "human_categories",
    "human_subcategories",
    "auto_categories",
    "auto_subcategories",
]:
    df[col] = df[col].apply(normalize_labels)

df.to_csv(CSV_PATH, index=False)

print(f"Updated CSV: {CSV_PATH}")
print(f"Matched images: {matched.sum()} / {len(df)}")
print(f"Unmatched images: {(~matched).sum()}")


In [ ]:
# Recheck label consistency

validation_df = pd.read_csv(OUT_CSV)
consistency_issues = check_label_consistency(validation_df)
display(consistency_issues)


In [ ]:
# Clean exclusive subcategories and refresh derived human labels

import shutil
from datetime import datetime, timezone

CSV_PATH = OUT_CSV
backup_path = CSV_PATH + datetime.now(timezone.utc).strftime(".%Y%m%dT%H%M%S%fZ.backup")
shutil.copy2(CSV_PATH, backup_path)
df = pd.read_csv(CSV_PATH)


def clean_subcategories(value):
    """Keep the most severe selected label in each exclusive taxonomy group."""
    labels = checked_subcategories(value)
    groups = [
        ["Unrealistic and Implied Violence", "Mild Violence", "Realistic Violence", "Gross Violence"],
        ["Mild Swearing", "Offensive Language"],
        ["Sexual Innuendo", "Mild Nudity Content", "Explicit Nudity Content", "Explicit Sexual Content"],
        ["Scary Content", "Moderately Disturbing Content", "Highly Disturbing Content"],
        ["Non-realistic Self-harm", "Self-injury", "Suicide"],
    ]
    names = {label.split(". ", 1)[-1] for label in labels}
    removed = set()
    for group in groups:
        selected = [name for name in group if name in names]
        if selected:
            removed.update(group[:group.index(selected[-1])])
    return [label for label in labels if label.split(". ", 1)[-1] not in removed]


skipped = []
for idx, value in df["human_subcategories"].items():
    try:
        labels = clean_subcategories(value)
    except ValueError as error:
        skipped.append((idx, str(error)))
        continue
    df.at[idx, "human_subcategories"] = serialize_labels(labels)
    df.at[idx, "human_categories"] = serialize_labels(derived_categories(labels))
    df.at[idx, "PEGI rating"] = get_max_pegi_score(labels)
df.to_csv(CSV_PATH, index=False)
print(f"Cleaned annotations; skipped {len(skipped)} invalid rows. Backup: {backup_path}")


## Compare annotations and export results


In [ ]:
# Compare human and automated annotations

import ast
import json
import os

import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
from llavaguard.taxonomy.PEGI.PEGI_Graph import get_max_pegi_score, get_rating

CSV_PATH = f"{local_data_dir}/evaluation/human_validation/human_validation.csv"
JSON_PATH = f"{local_data_dir}/datasets/PEGI-Guard-DS/v4/all_labels.json"

human_df = pd.read_csv(CSV_PATH)

with open(JSON_PATH, "r") as f:
    auto_data = json.load(f)

# Auto-generated data indexed by image path.
auto_by_img = {}

for x in auto_data:
    auto_image_path = (
        x.get("image_path")
        or x.get("image path")
        or x.get("image")
    )
    if auto_image_path is not None:
        auto_by_img[auto_image_path] = x


rows = []
unmatched_rows = []

for _, hrow in human_df.iterrows():
    image_path = hrow.get("image_path")

    arow = auto_by_img.get(image_path)
    if arow is None:
        unmatched_rows.append(hrow.to_dict())
        continue

    auto_id = arow.get("id")
    auto_image_path = (
        arow.get("image_path")
        or arow.get("image path")
        or arow.get("image")
    )

    human_cats = parse_categories(hrow.get("human_categories"))
    auto_cats = parse_categories(arow.get("categories"))

    human_subcats = parse_categories(hrow.get("human_subcategories"))
    auto_subcats = parse_categories(arow.get("subcategories"))

    human_pegi = normalize_pegi(hrow.get("PEGI rating"))
    auto_pegi = normalize_pegi(arow.get("PEGI rating"))

    rows.append(
        {
            # Updated id from all_labels.json.
            "id": auto_id,

            "image_path": image_path,
            "human_pegi": human_pegi,
            "auto_pegi": auto_pegi,
            "auto_subcategories": auto_subcats,
            "human_subcategories": human_subcats,
            "auto_categories": auto_cats,
            "human_categories": human_cats,

            "match": (
                human_pegi == auto_pegi
                if human_pegi is not None and auto_pegi is not None
                else None
            ),
            "age_diff": (
                auto_pegi - human_pegi
                if human_pegi is not None and auto_pegi is not None
                else None
            ),
            "model_too_strict": (
                auto_pegi > human_pegi
                if human_pegi is not None and auto_pegi is not None
                else None
            ),
            "model_too_lenient": (
                auto_pegi < human_pegi
                if human_pegi is not None and auto_pegi is not None
                else None
            ),
        }
    )

comparison_df = pd.DataFrame(rows, columns=["id", "image_path", "human_pegi", "auto_pegi", "auto_subcategories", "human_subcategories", "auto_categories", "human_categories", "match", "age_diff", "model_too_strict", "model_too_lenient"])
unmatched_df = pd.DataFrame(unmatched_rows)

valid_df = comparison_df.dropna(subset=["human_pegi", "auto_pegi"]).copy()

print("Human validation rows:", len(human_df))
print("Unmatched image paths:", len(unmatched_df))

if len(valid_df) == 0:
    print("\nNo rows had both human and automated PEGI ratings.")
    print("Human columns:", human_df.columns.tolist())
    print("Auto keys:", list(auto_data[0].keys()) if auto_data else [])
else:
    y_true = valid_df["human_pegi"].astype(int)
    y_pred = valid_df["auto_pegi"].astype(int)

    labels = [3, 7, 12, 16, 18]

    confusion_df = pd.DataFrame(
        confusion_matrix(y_true, y_pred, labels=labels),
        index=[f"human_{x}" for x in labels],
        columns=[f"auto_{x}" for x in labels],
    )

    report_df = pd.DataFrame(
        classification_report(
            y_true,
            y_pred,
            labels=labels,
            output_dict=True,
            zero_division=0,
        )
    ).T

    OUTPUT_DIR = f"{local_data_dir}/evaluation/human_validation/result"
    os.makedirs(OUTPUT_DIR, exist_ok=True)


    # Plot confusion matrix.
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(confusion_df.values)

    ax.set_xticks(range(len(confusion_df.columns)))
    ax.set_yticks(range(len(confusion_df.index)))
    ax.set_xticklabels(confusion_df.columns, rotation=45, ha="right")
    ax.set_yticklabels(confusion_df.index)

    ax.set_xlabel("Automated PEGI rating")
    ax.set_ylabel("Human PEGI rating")
    ax.set_title("PEGI Age Rating Confusion Matrix")

    for i in range(confusion_df.shape[0]):
        for j in range(confusion_df.shape[1]):
            ax.text(j, i, confusion_df.iloc[i, j], ha="center", va="center")

    fig.tight_layout()
    fig.savefig(
        os.path.join(OUTPUT_DIR, "age_rating_confusion_matrix.png"),
        dpi=300,
        bbox_inches="tight",
    )
    plt.close(fig)

    # Plot percentage confusion matrix.

    confusion_pct_df = confusion_df.div(confusion_df.sum(axis=1).replace(0, float("nan")), axis=0).fillna(0) * 100

    fig, ax = plt.subplots(figsize=(8, 6))

    im = ax.imshow(confusion_pct_df.values, vmin=0, vmax=100)

    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Percentage (%)")

    ax.set_xticks(range(len(confusion_pct_df.columns)))
    ax.set_yticks(range(len(confusion_pct_df.index)))

    ax.set_xticklabels(confusion_pct_df.columns, rotation=45, ha="right")
    ax.set_yticklabels(confusion_pct_df.index)

    ax.set_xlabel("Automated PEGI rating")
    ax.set_ylabel("Human PEGI rating")
    ax.set_title("Normalized PEGI Age Rating Confusion Matrix")

    for i in range(confusion_pct_df.shape[0]):
        for j in range(confusion_pct_df.shape[1]):
            value = confusion_pct_df.iloc[i, j]
            ax.text(
                j,
                i,
                f"{value:.1f}%",
                ha="center",
                va="center",
                color="white" if value > 50 else "black",
                fontsize=10,
            )

    fig.tight_layout()

    fig.savefig(
        os.path.join(
            OUTPUT_DIR,
            "age_rating_confusion_matrix_percentage.png",
        ),
        dpi=300,
        bbox_inches="tight",
    )

    plt.close(fig)

    report_df.to_csv(
        os.path.join(OUTPUT_DIR, "age_rating_per_rating.csv"),
    )


In [ ]:
# Calculate rating metrics

import json
from collections import defaultdict

RATINGS = [3, 7, 12, 16, 18]


def safe_div(a, b):
    return a / b if b else 0.0

def fbeta(precision, recall, beta):
    b2 = beta ** 2
    return safe_div((1 + b2) * precision * recall, (b2 * precision) + recall)

def round_metrics(d, ndigits=4):
    out = {}
    for k, v in d.items():
        if isinstance(v, float):
            out[k] = round(v, ndigits)
        elif isinstance(v, dict):
            out[k] = round_metrics(v, ndigits)
        else:
            out[k] = v
    return out


valid_df = comparison_df.dropna(subset=["human_pegi", "auto_pegi"]).copy()
valid_df["human_pegi"] = valid_df["human_pegi"].astype(int)
valid_df["auto_pegi"] = valid_df["auto_pegi"].astype(int)

invalid_list = comparison_df[
    comparison_df["human_pegi"].isna() | comparison_df["auto_pegi"].isna()
]["id"].tolist()

rating_confusion_matrix = defaultdict(lambda: defaultdict(int))

for _, row in valid_df.iterrows():
    rating_confusion_matrix[str(row["human_pegi"])][str(row["auto_pegi"])] += 1

rating_confusion_matrix = {
    str(true_rating): {
        str(pred_rating): rating_confusion_matrix[str(true_rating)].get(str(pred_rating), 0)
        for pred_rating in RATINGS
    }
    for true_rating in RATINGS
}

rating_per_label = {}

total_TP = total_FP = total_TN = total_FN = 0

for rating in RATINGS:
    y_true = valid_df["human_pegi"] == rating
    y_pred = valid_df["auto_pegi"] == rating

    TP = int((y_true & y_pred).sum())
    FP = int((~y_true & y_pred).sum())
    TN = int((~y_true & ~y_pred).sum())
    FN = int((y_true & ~y_pred).sum())

    total_TP += TP
    total_FP += FP
    total_TN += TN
    total_FN += FN

    TPR = safe_div(TP, TP + FN)
    FPR = safe_div(FP, FP + TN)
    FNR = safe_div(FN, TP + FN)
    TNR = safe_div(TN, TN + FP)
    precision = safe_div(TP, TP + FP)
    recall = TPR
    specificity = TNR
    f1 = fbeta(precision, recall, beta=1)
    f2 = fbeta(precision, recall, beta=2)

    rating_per_label[str(rating)] = {
        "Balanced Accuracy": safe_div(TPR + TNR, 2),
        "Accuracy": safe_div(TP + TN, TP + FP + TN + FN),
        "TP": TP,
        "FP": FP,
        "TN": TN,
        "FN": FN,
        "Invalid": 0,
        "TPR": TPR,
        "FPR": FPR,
        "FNR": FNR,
        "TNR": TNR,
        "Precision": precision,
        "Recall": recall,
        "Specificity": specificity,
        "F1": f1,
        "F2": f2,
        "Invalid_list": [],
    }

TPR = safe_div(total_TP, total_TP + total_FN)
FPR = safe_div(total_FP, total_FP + total_TN)
FNR = safe_div(total_FN, total_TP + total_FN)
TNR = safe_div(total_TN, total_TN + total_FP)
precision = safe_div(total_TP, total_TP + total_FP)
recall = TPR

results = {
    "Accuracy": safe_div(int((valid_df["human_pegi"] == valid_df["auto_pegi"]).sum()), len(valid_df)),
    "One-vs-Rest Balanced Accuracy": safe_div(TPR + TNR, 2),
    "One-vs-Rest Accuracy": safe_div(total_TP + total_TN, total_TP + total_FP + total_TN + total_FN),
    "number_of_samples": int(len(valid_df)),
    "total_TP": int(total_TP),
    "total_FP": int(total_FP),
    "total_TN": int(total_TN),
    "total_FN": int(total_FN),
    "total_invalid": int(len(invalid_list)),
    "TPR": TPR,
    "FPR": FPR,
    "FNR": FNR,
    "TNR": TNR,
    "Precision": precision,
    "Total F1": fbeta(precision, recall, beta=1),
    "Total F2": fbeta(precision, recall, beta=2),
    "rating_per_label": rating_per_label,
    "rating_confusion_matrix": rating_confusion_matrix,
    "invalid_list": invalid_list,
    "ratings_over_rate": int((valid_df["auto_pegi"] > valid_df["human_pegi"]).sum()),
    "ratings_under_rate": int((valid_df["auto_pegi"] < valid_df["human_pegi"]).sum()),
}

results = round_metrics(results)

OUTPUT_PATH = f"{local_data_dir}/evaluation/human_validation/human_vs_auto_pegi_age_stats.json"

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
with open(OUTPUT_PATH, "w") as f:
    json.dump(results, f, indent=4)

print(f"Saved stats to: {OUTPUT_PATH}")


In [ ]:
# Plot annotation distributions

import ast
import os
import re

import matplotlib.pyplot as plt
import pandas as pd

# Plots distribution of human validation labels over subcategories and super-categories.

CSV_PATH = f"{local_data_dir}/evaluation/human_validation/human_validation.csv"
OUTPUT_DIR = f"{local_data_dir}/evaluation/human_validation/result"

os.makedirs(OUTPUT_DIR, exist_ok=True)


def parse_label_list(value):
    """Keep missing annotations distinct from explicit safe labels in plots."""
    if is_missing(value):
        return ["Missing annotation"]
    return sorted(set(parse_categories(value))) or ["NA: None applying"]


def subcategory_sort_key(label):
    if label == "NA: None applying":
        return float("inf")

    match = re.match(r"^\s*(\d+)", str(label))
    return int(match.group(1)) if match else float("inf")


def category_sort_key(label):
    if label == "NA: None applying":
        return float("inf")

    match = re.match(r"^\s*S(\d+)", str(label))
    return int(match.group(1)) if match else float("inf")

df = pd.read_csv(CSV_PATH)

# Human subcategory distribution.

all_subcategories = get_content_categories()

human_subcategories = (
    df["human_subcategories"]
    .apply(parse_label_list)
    .explode()
    .astype(str)
    .str.strip()
)

subcategory_counts = human_subcategories.value_counts()

# Include all taxonomy subcategories, even when their count is zero.
subcategory_index = list(all_subcategories)

for label in ["NA: None applying", "Missing annotation"]:
    if label in subcategory_counts.index:
        subcategory_index.append(label)

subcategory_counts = (
    subcategory_counts
    .reindex(subcategory_index, fill_value=0)
    .sort_index(key=lambda index: index.map(subcategory_sort_key))
)

subcategory_counts_df = subcategory_counts.to_frame(name="image_count")

fig_height = max(8,len(subcategory_counts_df) * 0.35)

fig, ax = plt.subplots(figsize=(13, fig_height))

bars = ax.barh(
    subcategory_counts_df.index,
    subcategory_counts_df["image_count"],
)

ax.set_xlabel("Number of evaluated images")
ax.set_ylabel("Human safety subcategory")
ax.set_title(
    "Distribution of Evaluated Images by Human Safety Subcategory"
)

ax.invert_yaxis()
ax.grid(axis="x", alpha=0.3)

max_subcategory_count = subcategory_counts_df["image_count"].max()

label_offset = (
    max_subcategory_count * 0.01
    if max_subcategory_count > 0
    else 0.1
)

for bar, value in zip(bars, subcategory_counts_df["image_count"]):
    ax.text(
        bar.get_width() + label_offset,
        bar.get_y() + bar.get_height() / 2,
        str(int(value)),
        va="center",
    )

ax.set_xlim(0, max_subcategory_count * 1.12 if max_subcategory_count > 0 else 1)

fig.tight_layout()

fig.savefig(
    os.path.join(OUTPUT_DIR,"human_subcategory_distribution.png"),
    dpi=300,
    bbox_inches="tight",
)

plt.close(fig)

# Human super-category distribution.

all_categories = get_content_super_categories()

human_categories = (
    df["human_categories"]
    .apply(parse_label_list)
    .explode()
    .astype(str)
    .str.strip()
)

category_counts = human_categories.value_counts()

# Include all taxonomy super-categories, even when count is zero.
category_index = list(all_categories)

for label in ["NA: None applying", "Missing annotation"]:
    if label in category_counts.index:
        category_index.append(label)

category_counts = (
    category_counts
    .reindex(category_index, fill_value=0)
    .sort_index(key=lambda index: index.map(category_sort_key))
)

category_counts_df = category_counts.to_frame(name="image_count")

fig_height = max(6,len(category_counts_df) * 0.65,)

fig, ax = plt.subplots(figsize=(12, fig_height))

bars = ax.barh(
    category_counts_df.index,
    category_counts_df["image_count"],
)

ax.set_xlabel("Number of evaluated images")
ax.set_ylabel("Human safety category")
ax.set_title( "Distribution of Evaluated Images by Human Safety Category")

ax.invert_yaxis()
ax.grid(axis="x", alpha=0.3)

max_category_count = category_counts_df["image_count"].max()

label_offset = (max_category_count * 0.01 if max_category_count > 0 else 0.1)

for bar, value in zip(bars, category_counts_df["image_count"]):
    ax.text(
        bar.get_width() + label_offset,
        bar.get_y() + bar.get_height() / 2,
        str(int(value)),
        va="center",
    )

ax.set_xlim(0, max_category_count * 1.12 if max_category_count > 0 else 1)

fig.tight_layout()

fig.savefig(
    os.path.join(OUTPUT_DIR,"human_category_distribution.png"),
    dpi=300,
    bbox_inches="tight",
)

plt.close(fig)

subcategory_counts_df.to_csv(os.path.join(OUTPUT_DIR, "human_subcategory_counts.csv"))
category_counts_df.to_csv(os.path.join(OUTPUT_DIR, "human_category_counts.csv"))
print("Saved plots and count tables to:", OUTPUT_DIR)
